# 1.Feature Engineering

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

Load Dataset

In [13]:
# Load cleaned dataset 
df = pd.read_csv('cleaned_diabetes.csv')

In [14]:
!pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.0 MB 5.1 MB/s eta 0:00:02
   ---------- ----------------------------- 2.1/8.0 MB 5.5 MB/s eta 0:00:02
   ----------------- ---------------------- 3.4/8.0 MB 5.9 MB/s eta 0:00:01
   ----------------------- ---------------- 4.7/8.0 MB 6.0 MB/s eta 0:00:01
   ------------------------------- -------- 6.3/8.0 MB 6.1 MB/s eta 0:00:01
   ------------------------------------ --- 7.3/8.0 MB 6.2 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 6.1 MB/s  0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   -------- ------------------------------- 1/5 [joblib]
   -------- ------------

In [15]:
from imblearn.over_sampling import SMOTE

# Balance the training data using SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_pca, y_train)

print(f"Original training target distribution:\n{y_train.value_counts()}")
print(f"Resampled training target distribution:\n{y_train_resampled.value_counts()}")

ModuleNotFoundError: No module named 'imblearn'

Split into 80% training data and 20% testing data

In [ ]:
# Split into 80% training data and 20% testing data\
X = df.drop(columns=['readmitted'])
y = df['readmitted']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Scale the features

In [ ]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Apply PCA

In [ ]:
# Apply PCA to keep the top 5 most important components
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Original dimensions: {X_train_scaled.shape[1]}")
print(f"Dimensions kept after PCA: {pca.n_components_}")

# 2.Random Forest Algorithm

In [ ]:
from sklearn.ensemble import RandomForestClassifier

Initialize and train the Random Forest model

In [ ]:
# Initialize an optimized Random Forest model
# max_depth prevents the trees from growing endlessly (overfitting)
# class_weight ensures the model respects all classes equally
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15, 
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1 # Uses all your computer's CPU cores to train faster!
)

# Train the model on the balanced data!
rf_model.fit(X_train_resampled, y_train_resampled)

Make predictions on the test set

In [ ]:
predictions = rf_model.predict(X_test_pca)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Print the overall accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"Overall Test Accuracy: {accuracy:.4f}\n")

# Print the detailed classification report
print("Classification Report:")
print(classification_report(y_test, predictions))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Generate the confusion matrix
cm = confusion_matrix(y_test, predictions)

# Plot it using Seaborn for a clean, professional look
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Class 0', 'Class 1', 'Class 2'], 
            yticklabels=['Class 0', 'Class 1', 'Class 2'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Random Forest Confusion Matrix')
plt.show()